# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZaraTrimizi/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will start with Logistic Regression because the target is binary: `is_declining_label` indicates whether a content item is observed as declining (`1`) or not (`0`).

The model is also suitable for my lane because the goal is to rank content items for review. Logistic Regression provides a predicted probability for each item, which can be used as a ranking score.

I chose Logistic Regression as the first model because it is simple, interpretable, and provides a clear comparison with my Week-4 rule-based baseline. I will only add a more complex model if it provides a meaningful improvement on the same evaluation data and metric.

In [57]:
!git clone https://github.com/samana-gillani/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 173 (delta 78), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 1.88 MiB | 11.91 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [58]:
import sys
sys.path.append("/content/flyrank-ml-internship/scripts")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

from ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES,
    precision_at_k,
    normalize,
)

RANDOM_STATE = 42

print("Numeric features:", len(MODEL_NUMERIC_FEATURES))
print("Categorical features:", len(MODEL_CATEGORICAL_FEATURES))
print("Random seed:", RANDOM_STATE)

Numeric features: 18
Categorical features: 8
Random seed: 42


## 2. Split design

I will use a client-grouped train/test split so that content items from the same client do not appear in both training and test sets.

This is more honest than randomly splitting individual content items because pages belonging to the same client can share characteristics. Grouping by `client_id` reduces the risk that the model benefits from seeing the same client's patterns during training and testing.

I will use a fixed random seed so that the split is reproducible.

In [59]:
from pathlib import Path

raw_path = Path("data/raw/content_refresh_anonymized.csv")

print("Raw dataset exists:", raw_path.exists())

if raw_path.exists():
    print("Path:", raw_path)

Raw dataset exists: True
Path: data/raw/content_refresh_anonymized.csv


In [60]:
!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


In [61]:
# Load the prepared modeling dataset
DATA_PATH = "data/processed/refresh_feature_vector.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

print("\nUnique clients:", df["client_id"].nunique())

Rows: 30000
Columns: 52

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667

Unique clients: 32


In [62]:
# Create a client-grouped train/test split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"],
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("\nClient overlap:", len(overlap))

assert len(overlap) == 0, "Client leakage detected!"

print("\nTrain declining rate:", train_df["is_declining_label"].mean())
print("Test declining rate:", test_df["is_declining_label"].mean())

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7

Client overlap: 0

Train declining rate: 0.5501111717078492
Test declining rate: 0.5109524582184002


## 3. Train + compare vs my baseline
I train a Logistic Regression model using the same prepared dataset, client-based train/test split, and evaluation metric used for the Week-4 baseline.

The model predicts the probability that a page is declining. I compare its ranking performance against the Week-4 baseline using **Precision@K** at K = 20 and K = 50.

Using the same test rows and the same metric makes the comparison fair and shows whether the machine learning model improves the ability to prioritize genuinely declining pages for review.

In [63]:
# Define features and target
TARGET = "is_declining_label"

X_train = train_df[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES]
y_train = train_df[TARGET]

X_test = test_df[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES]
y_test = test_df[TARGET]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training features:", X_train.shape[1])
print("Test features:", X_test.shape[1])

Training rows: 23837
Test rows: 6163
Training features: 26
Test features: 26


In [64]:
# Preprocessing for numeric and categorical features
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, MODEL_NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, MODEL_CATEGORICAL_FEATURES),
    ]
)

In [65]:
# Logistic Regression model
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression training complete.")

Logistic Regression training complete.


In [66]:
# Predicted probability of the declining class
model_scores = logistic_model.predict_proba(X_test)[:, 1]

print("Generated scores:", len(model_scores))
print("Minimum score:", model_scores.min())
print("Maximum score:", model_scores.max())
print("Mean score:", model_scores.mean())

Generated scores: 6163
Minimum score: 0.0284448386331258
Maximum score: 0.9574242581248165
Mean score: 0.5566452093883577


In [67]:
# ============================================================
# Week-4 baseline reproduced on the Week-5 test set
# ============================================================

baseline_test = test_df.copy()

# Week-4 baseline uses:
# 1. normalized impressions
# 2. CTR priority (lower CTR = higher priority)
# 3. freshness (older content = higher priority)
#
# Calculate normalization using ONLY the Week-5 test set so that
# the baseline is evaluated on exactly the same held-out rows.

def min_max_normalize(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)

    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(0.0, index=series.index)

    return (series - minimum) / (maximum - minimum)


# 1. Visibility / impressions
baseline_test["impression_score"] = min_max_normalize(
    baseline_test["impressions_90d"]
)

# 2. Low CTR gets higher priority
baseline_test["ctr_priority"] = 1 - min_max_normalize(
    baseline_test["ctr"]
)

# 3. Older content gets higher priority
baseline_test["freshness_score"] = min_max_normalize(
    baseline_test["days_since_last_update"]
)

# Week-4 scoring rule
baseline_test["baseline_score"] = (
    baseline_test["impression_score"] * 0.40
    + baseline_test["ctr_priority"] * 0.30
    + baseline_test["freshness_score"] * 0.30
)

print("Baseline scores generated:", len(baseline_test))

display(
    baseline_test[
        [
            "content_id",
            "is_declining_label",
            "baseline_score",
            "impressions_90d",
            "ctr",
            "days_since_last_update",
        ]
    ]
    .sort_values("baseline_score", ascending=False)
    .head(10)
)

Baseline scores generated: 6163


,content_id,is_declining_label,baseline_score,impressions_90d,ctr,days_since_last_update
6653,content_5fe46e04994d,1,0.948320,517715,0.14,104
21565,content_9532f197bbc8,1,0.780638,309192,0.87,104
26844,content_8c19996aa890,1,0.729926,509252,0.15,20
21819,content_4c36c775b818,1,0.691930,463103,0.41,20
23446,content_1aa219431528,0,0.664594,151541,0.23,104
19645,content_3ad3a781fa91,0,0.651923,145044,1.08,104
26564,content_e6e15ac13287,0,0.645419,128704,0.40,104
29716,content_fac19fcdfb85,0,0.645152,126611,0.25,104
2365,content_4a18c07e5357,1,0.638455,125049,0.86,104
17400,content_e5ae436f9a16,0,0.636499,117741,0.45,104


In [68]:
# Attach Logistic Regression probabilities to the same test rows
baseline_test = baseline_test.copy()

baseline_test["predicted_probability"] = model_scores

print("Rows:", len(baseline_test))
print("Model scores:", len(model_scores))

display(
    baseline_test[
        ["content_id", "is_declining_label",
         "baseline_score", "predicted_probability"]
    ].head()
)

Rows: 6163
Model scores: 6163


,content_id,is_declining_label,baseline_score,predicted_probability
0,content_304f48230142,1,0.333912,0.599635
1,content_a1fb4e703a9e,1,0.361806,0.758355
5,content_d4084a4bc775,1,0.340612,0.852547
13,content_a5a2fbc76336,0,0.547295,0.737550
19,content_af865035b328,1,0.319709,0.669249


In [69]:
# ============================================================
# Model vs Week-4 baseline
# Same test rows + same metric
# ============================================================

comparison_rows = []

for k in [20, 50]:

    baseline_precision = precision_at_k(
        baseline_test["is_declining_label"],
        baseline_test["baseline_score"],
        k
    )

    model_precision = precision_at_k(
        baseline_test["is_declining_label"],
        baseline_test["predicted_probability"],
        k
    )

    comparison_rows.append({
        "K": k,
        "Baseline Precision@K": round(baseline_precision, 3),
        "Logistic Regression Precision@K": round(model_precision, 3),
        "Test base rate": round(
            baseline_test["is_declining_label"].mean(), 3
        )
    })

comparison = pd.DataFrame(comparison_rows)

display(comparison)

,K,Baseline Precision@K,Logistic Regression Precision@K,Test base rate
0,20,0.30,0.70,0.511
1,50,0.32,0.72,0.511


In [70]:
base_rate = y_test.mean()

comparison["Test base rate"] = base_rate

comparison

,K,Baseline Precision@K,Logistic Regression Precision@K,Test base rate
0,20,0.30,0.70,0.510952
1,50,0.32,0.72,0.510952


### Comparison Result

The Logistic Regression model outperforms the Week-4 baseline at both evaluation cutoffs. Precision@20 improves from 0.30 to 0.70, while Precision@50 improves from 0.32 to 0.72.

This shows that the learned model is better at ranking pages that are actually declining near the top of the review queue. The comparison uses the same client-held-out test set and the same Precision@K metric, making the result directly comparable to the Week-4 baseline.

## 4. Errors and interpretation

The largest positive Logistic Regression coefficients were associated with `log_impressions_90d`, the low impression tier, the 1000–2000 word-count tier, unknown main intent, and word count. The largest negative coefficients were associated with the top-3 position tier, excellent impression tier, `log_clicks_90d`, unknown competition level, and the 31–90 day freshness tier.

These coefficients describe associations used by the fitted model rather than causal effects. The strongest signals suggest that the model is combining search visibility, engagement, content characteristics, position, and freshness information when estimating decline probability.

The feature set does not include `trend_direction` or `trend_pct`, which are excluded because they are directly related to the target definition. Therefore, these target-derived fields were not used as model inputs.

In [71]:
# Build a table of test predictions for error analysis

error_df = test_df.copy()

error_df["predicted_probability"] = model_scores
error_df["predicted_label"] = (model_scores >= 0.50).astype(int)

error_df["correct"] = (
    error_df["predicted_label"] == error_df["is_declining_label"]
)

error_df["error_type"] = np.select(
    [
        (error_df["is_declining_label"] == 1) & (error_df["predicted_label"] == 0),
        (error_df["is_declining_label"] == 0) & (error_df["predicted_label"] == 1),
    ],
    [
        "false_negative",
        "false_positive",
    ],
    default="correct",
)

print("Total test rows:", len(error_df))
print("\nPrediction counts:")
print(error_df["error_type"].value_counts())

print("\nAccuracy:")
print(error_df["correct"].mean())

Total test rows: 6163

Prediction counts:
error_type
correct           3620
false_positive    1675
false_negative     868
Name: count, dtype: int64

Accuracy:
0.5873762777867921


In [72]:
# Show the most confident false positives and false negatives

false_positives = (
    error_df[error_df["error_type"] == "false_positive"]
    .sort_values("predicted_probability", ascending=False)
)

false_negatives = (
    error_df[error_df["error_type"] == "false_negative"]
    .sort_values("predicted_probability", ascending=True)
)

print("Top false positives:")
display(
    false_positives[
        [
            "content_id",
            "is_declining_label",
            "predicted_probability",
            "content_type",
            "impressions_90d",
            "ctr",
            "avg_position",
            "content_age_days",
        ]
    ].head(3)
)

print("\nTop false negatives:")
display(
    false_negatives[
        [
            "content_id",
            "is_declining_label",
            "predicted_probability",
            "content_type",
            "impressions_90d",
            "ctr",
            "avg_position",
            "content_age_days",
        ]
    ].head(3)
)

Top false positives:


,content_id,is_declining_label,predicted_probability,content_type,impressions_90d,ctr,avg_position,content_age_days
26614,content_7be5f150dc65,0,0.954100,keyword article,290,0.0,5.9,96
20736,content_41baf0722ad9,0,0.942681,keyword article,3115,0.0,12.8,275
12869,content_5d5653c4eb4f,0,0.932321,keyword article,15101,0.0,5.7,421



Top false negatives:


,content_id,is_declining_label,predicted_probability,content_type,impressions_90d,ctr,avg_position,content_age_days
8407,content_d1e915d03c28,1,0.063800,keyword article,2,0.0,45.0,537
29158,content_e18144cbd19d,1,0.066531,keyword article,3,0.0,2.0,545
17690,content_c268b1716236,1,0.079421,keyword article,3,0.0,41.7,502


In [73]:
# Extract transformed feature names and Logistic Regression coefficients

fitted_preprocessor = logistic_model.named_steps["preprocessor"]
classifier = logistic_model.named_steps["classifier"]

feature_names = fitted_preprocessor.get_feature_names_out()

coefficients = classifier.coef_[0]

importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients),
    }
).sort_values("absolute_coefficient", ascending=False)

print("Top 10 features by absolute coefficient:")
display(importance_df.head(10))

Top 10 features by absolute coefficient:


,feature,coefficient,absolute_coefficient
5,numeric__log_impressions_90d,1.437537,1.437537
51,categorical__position_tier_top_3,-0.800444,0.800444
45,categorical__impression_tier_low,0.743427,0.743427
38,categorical__word_count_tier_1000-2000,0.614016,0.614016
43,categorical__impression_tier_excellent,-0.610980,0.610980
6,numeric__log_clicks_90d,-0.562502,0.562502
29,categorical__main_intent_unknown,0.519570,0.519570
3,numeric__word_count,0.502218,0.502218
24,categorical__content_type_keyword article,0.461645,0.461645
21,categorical__competition_level_unknown,-0.448640,0.448640


In [74]:
# Show the strongest positive and negative Logistic Regression coefficients

print("Features associated with higher predicted decline probability:")
display(
    importance_df
    .sort_values("coefficient", ascending=False)
    [["feature", "coefficient"]]
    .head(5)
)

print("\nFeatures associated with lower predicted decline probability:")
display(
    importance_df
    .sort_values("coefficient", ascending=True)
    [["feature", "coefficient"]]
    .head(5)
)

Features associated with higher predicted decline probability:


,feature,coefficient
5,numeric__log_impressions_90d,1.437537
45,categorical__impression_tier_low,0.743427
38,categorical__word_count_tier_1000-2000,0.614016
29,categorical__main_intent_unknown,0.519570
3,numeric__word_count,0.502218



Features associated with lower predicted decline probability:


,feature,coefficient
51,categorical__position_tier_top_3,-0.800444
43,categorical__impression_tier_excellent,-0.610980
6,numeric__log_clicks_90d,-0.562502
21,categorical__competition_level_unknown,-0.448640
36,categorical__freshness_tier_31-90,-0.390136


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.